# overlays

> Popovers, menus, dialogs, drawers, and tooltips

In [ ]:
#| default_exp overlays

An overlay sits above the page without changing its layout. Popovers close on a click outside them, which the platform calls light dismiss. Modal dialogs make the page behind them inert. We keep those native behaviours, add a shared floating surface, and position popovers beside the control that opened them. Dropdown menus are popovers too. Tooltips use the same anchor positioning, but they are not popovers.

In [ ]:
from fasthtml.common import Div, Span, Button, A, P, Small, H2, Hr, FT, unqid
from answerui.registry import reg, hypelt
from answerui.layout import Stack, Cluster
from answerui.demo import example
import answerui.base  # the reset and tokens this styling is written on
import answerui.controls  # the buttons and menu items the panels hold


In [ ]:
from fasthtml.common import *
from answerui.demo import demo
from fastcore.test import *


## Popover

`Popover` is a floating panel on the raised neutral surface, `--n-2`. A button opens it with `popovertarget`, and the browser closes it on light dismiss or Escape. When the panel opens or closes, the `expanded` rule sets `aria-expanded` on every button that opens or toggles it, and `Button`'s CSS draws a button with `aria-expanded="true"` as pressed. The attribute is for the stylesheet: assistive technology already gets the open state from `popovertarget`. `Popover` also sets an `anchored` attribute. Its CSS applies only to `[popover][anchored]`, not to other elements with the native `popover` attribute.

CSS anchor positioning places the panel. `position-area` puts the panel below the button that opened it, aligned to the button's start edge. Neither element needs an anchor name, because a popover opened by `popovertarget` or `commandfor` has that button as its implicit anchor. If the panel would overflow the viewport, `position-try-fallbacks` flips it above the button, aligns it to the button's end edge instead, or both. If the panel is taller than the space both above and below the button, `position-try-order` puts it on the roomier side, where it scrolls within that space less a 4px gap and an 8px viewport margin. To open a menu below a navbar rather than across the navbar's padding, mark the navbar with `anchor`. That makes the navbar a named anchor, and each panel inside it is placed with its top at the navbar's bottom edge. The browser keeps the panel in place as the page scrolls, the window resizes or the content changes size, with no script. In a browser without anchor positioning, the `placement` rule's script puts the panel below or above the button, keeps it inside the viewport, and moves it as the page scrolls or resizes. In browsers with anchor positioning, the script returns at once.

In [ ]:
_fallback = "if (CSS.supports('position-area', 'center')) return;\n"
_popover_on = {
    ('button[popovertarget], button[commandfor]', 'click'): _fallback + r"""
const id = el.getAttribute('popovertarget') || el.getAttribute('commandfor');
const panel = document.getElementById(id);
if (panel?.matches('[popover][anchored]')) panel._anchor = el;""",
    ('&', 'beforetoggle', True): _fallback + r"""
if (e.target !== el) return;
if (e.newState === 'open') {
    if (e.source) el._anchor = e.source;
    el.style.visibility = 'hidden';
} else el._unwatch?.();""",
    ('&', 'toggle', True): _fallback + r"""
if (e.target !== el) return;
el._unwatch?.();
if (!el.matches(':popover-open')) return;
const anchor = el._anchor;
const place = () => {
    if (!anchor?.isConnected) { el.hidePopover(); return; }
    const r = anchor.getBoundingClientRect(), b = (anchor.closest('[anchor]') ?? anchor).getBoundingClientRect(), gap = 4, edge = 8;
    const vw = document.documentElement.getBoundingClientRect().width, vh = document.documentElement.clientHeight;
    const below = vh - b.bottom - gap - edge, above = b.top - gap - edge;
    const height = el.scrollHeight + el.offsetHeight - el.clientHeight;
    const up = height > below && above > below;
    el.style.setProperty('--_available-height', Math.max(0, up ? above : below) + 'px');
    const x = getComputedStyle(anchor).direction === 'rtl' ? r.right - el.offsetWidth : r.left;
    const y = up ? b.top - gap - el.offsetHeight : b.bottom + gap;
    el.style.left = Math.max(edge, Math.min(x, vw - el.offsetWidth - edge)) + 'px';
    el.style.top = Math.max(edge, Math.min(y, vh - el.offsetHeight - edge)) + 'px';
};
if (anchor) {
    place();
    const watch = new AbortController(), size = new ResizeObserver(place);
    addEventListener('resize', place, {signal: watch.signal});
    document.addEventListener('scroll', place, {capture: true, signal: watch.signal});
    size.observe(anchor); size.observe(el);
    el._unwatch = () => { watch.abort(); size.disconnect(); };
}
el.style.visibility = '';""",
    ('&', 'cleanup'): _fallback + "if (e.target === el) el._unwatch?.()"}

_popover_css = r"""
--face: var(--n-2); background: var(--face); color: var(--n2);
border: 1px solid var(--n-1); border-radius: var(--radius); padding: var(--space); box-shadow: var(--floating);
position: fixed; inset: auto; margin: 0; max-inline-size: calc(100vw - 2 * var(--space)); overflow: auto;
position-area: block-end span-inline-end; position-try-fallbacks: flip-block, flip-inline, flip-block flip-inline; position-try-order: most-block-size;
margin-block-start: 4px; max-block-size: calc(100% - 12px);
[anchor] & { inset-block-start: anchor(--anchor bottom); }
@supports not (position-area: center) {
  & { margin: 0; max-block-size: var(--_available-height, calc(100dvh - 2 * var(--space))); }
}
"""

@hypelt(sel='[popover][anchored]', layer='components', css=_popover_css)
def Popover(*c, popover='auto', **attrs):
    "A floating panel positioned beside the button that opens it"
    return Div(*c, popover=popover, anchored=True, **attrs)

reg.rule('[anchor] { anchor-name: --anchor; }', layer='components')

Popover.rule('expanded', on={('[popover]', 'toggle', True): r"""
if (e.target !== el) return;
for (const b of document.querySelectorAll(`[popovertarget="${CSS.escape(el.id)}"]:not([popovertargetaction=hide]), [commandfor="${CSS.escape(el.id)}"]:is([command=toggle-popover], [command=show-popover])`)) b.setAttribute('aria-expanded', String(el.matches(':popover-open')));"""},
    layer='components', doc='Every button that opens or toggles the panel carries `aria-expanded` while it is open.')
Popover.rule('placement', on=_popover_on, layer='components', doc='In an engine without anchor positioning: remembers the button that opened the panel, places the panel beside it within the viewport, and follows scrolling and resizing until it closes.')

Popover.placement [components] [popover][anchored]: In an engine without anchor positioning: remembers the button that opened the panel, places the panel beside it within the viewport, and follows scrolling and resizing until it closes.

The panel fades in as it opens and out as it closes. Transitions on `display` and `overlay` keep the panel rendered in the top layer until the fade ends. The `fade` rule fades dialogs and their backdrops the same way:


In [ ]:
reg.rule(r'''
:is(dialog, [popover][anchored]), dialog::backdrop {
  opacity: 0;
  transition: opacity var(--motion) var(--ease), translate var(--motion) var(--ease), display var(--motion) allow-discrete, overlay var(--motion) allow-discrete;
}
:is(dialog[open], [popover][anchored]:popover-open), dialog[open]::backdrop { opacity: 1; @starting-style { opacity: 0; } }
''', name='fade', layer='components', doc='Dialogs, their backdrops, and anchored popovers fade over `--motion`, staying in the top layer until the fade ends.')

fade [components]: Dialogs, their backdrops, and anchored popovers fade over `--motion`, staying in the top layer until the fade ends.

Any button can open a popover by setting `popovertarget` to the popover's id. The panel holds ordinary content:

In [ ]:
@example(Popover)
def _popover_example():
    id = unqid()
    return Cluster(Button('What is a workspace?', type='button', secondary=True, popovertarget=id),
        Popover(P("A workspace holds a team's projects, members, and billing."), id=id, style='max-inline-size: 18rem'))

In [ ]:
Popover.demo(height='9rem')

```html
<div cluster>
<button type="button" secondary popovertarget="_yk_lWXZtQlqFjCAxpdrqkw">What is a workspace?</button>  <div popover="auto" anchored id="_yk_lWXZtQlqFjCAxpdrqkw" style="max-inline-size: 18rem">
    <p>A workspace holds a team's projects, members, and billing.</p>
  </div>
</div>

```

## Dropdown

`Dropdown` returns a trigger and its popover of links, buttons, and separators. Other content, such as a heading with the signed-in user's name, is indented to line up with the items' text. Give it text for a secondary trigger, or a configured `Button` to choose the appearance, accessible name, and other attributes. A trigger that shows only an avatar or an icon needs a name of its own, so pass a `Button` with an `aria-label`. Clicking the trigger never submits a surrounding form, because `Dropdown` always sets its `type` to `button`.

These are ordinary HTML controls, not an ARIA application menu: Tab and Shift+Tab move between the items, and a disabled item is skipped because it cannot take focus. When the panel opens, the browser focuses the first enabled item, which `Dropdown` marks `autofocus`. Clicking a button in the panel closes the panel, because `Dropdown` gives each button without a `popovertarget` of its own `popovertarget` set to the panel and `popovertargetaction="hide"`. The `choose` rule closes the panel when a link in it is clicked, whether the browser or htmx follows the link. The browser still handles Escape and light dismiss. The `disabled` rule cancels any click on an item marked `aria-disabled="true"`, and stops it reaching other handlers.

In [ ]:
_dropdown_disabled = {('& > [aria-disabled=true]', 'click', True): "e.preventDefault(); e.stopImmediatePropagation()"}

_dropdown_css = r"""
min-inline-size: min(12rem, calc(100vw - 2 * var(--space))); padding: var(--space-3);
&:popover-open { display: grid; align-content: start; block-size: max-content; }
"""

@hypelt(sel='[popover][dropdown]', layer='components', css=_dropdown_css)
def Dropdown(
    trigger,  # Label text or a configured Button
    *items,   # Buttons, links, Hr separators, and other content such as a heading, which aligns with the items
    id=None,  # Panel id; generated when omitted
    **attrs,  # Attributes for the panel
):
    "A button and its popover of ordinary links and actions"
    id = id or unqid()
    if not isinstance(trigger, FT) or trigger.tag != 'button': trigger = Button(trigger, secondary=True)
    acts = [c for c in items if isinstance(c, FT) and c.tag in ('a', 'button')]
    for c in acts:
        if c.tag == 'button' and 'popovertarget' not in c.attrs: c(popovertarget=id, popovertargetaction='hide')
    live = [c for c in acts if (c.get('href') if c.tag == 'a' else not c.get('disabled')) and c.get('aria-disabled') != 'true' and not c.get('hidden')]
    if live: live[0](autofocus=True)
    return trigger(type='button', popovertarget=id), Popover(*items, dropdown=True, id=id, **attrs)

Dropdown.rule('disabled', on=_dropdown_disabled, layer='components', doc='An item marked aria-disabled does nothing.')
Dropdown.rule('choose', on={('& > a[href]', 'click'): "el.closest('[popover]').hidePopover()"}, layer='components',
    doc='Following a link closes the dropdown; a button closes it natively through popovertargetaction.')

reg.rule(r'''
[dropdown] {
  > :is(a, button) { display: block; inline-size: 100%; text-align: start; padding: var(--space-2) var(--space-1);
    font: inherit; font-weight: var(--weight); color: inherit; text-decoration: none;
    background: none; border: 0; border-radius: var(--radius-1); box-shadow: none;
    &:not(:disabled, [aria-disabled=true]):is(:hover, :focus-visible) { background: var(--v-3); color: var(--v2); }
    &[danger]:not(:disabled, [aria-disabled=true]) { color: var(--v); }
    &:is(:disabled, [aria-disabled=true]) { color: var(--n1); cursor: default; } }
  > hr { margin-block: var(--space-3); }
  > :not(a, button, hr) { padding-inline: var(--space-1); } }
''', layer='components')

c8ec90d7f7bc [components]

Here the first enabled item, the Profile link, gets `autofocus`, and each button gets `popovertargetaction="hide"`:

In [ ]:
trigger, panel = Dropdown('Account', Button('Duplicate', type='button', disabled=True), A('Profile', href='#'), Hr(), Button('Sign out', type='button'))
test_eq([x.get('autofocus') for x in panel.children], [None, True, None, None])
test_eq([x.get('popovertargetaction') for x in panel.children], ['hide', None, None, 'hide'])

The Account menu starts with the user's name and email, lined up with its items, and uses the default trigger. The Options menu passes a configured `Button` as its trigger. Tab through the items: the heading is skipped, and so is the disabled Duplicate action.

In [ ]:
@example(Dropdown)
def _dropdown_example():
    trigger, panel = Dropdown(Button('Options', aria_label='Project actions'),
        Button('Rename', type='button'), Button('Duplicate', type='button', disabled=True),
        Hr(), Button('Delete', type='button', danger=True))
    assert trigger.get('aria-label') == 'Project actions'
    return Cluster(Dropdown('Account', P('Ada Lovelace'), Small('ada@example.com'), Hr(), A('Profile', href='#'), A('Billing', href='#'),
        Hr(), Button('Sign out', type='button')), trigger, panel)

In [ ]:
Dropdown.demo(height='14rem')

```html
<div cluster>
<button secondary type="button" popovertarget="_Xa76tJBKQJ2sN_E5ZXO8Nw">Account</button>  <div popover="auto" anchored dropdown id="_Xa76tJBKQJ2sN_E5ZXO8Nw">
    <p>Ada Lovelace</p>
<small>ada@example.com</small><hr><a href="#" autofocus>Profile</a><a href="#">Billing</a><hr><button type="button" popovertarget="_Xa76tJBKQJ2sN_E5ZXO8Nw" popovertargetaction="hide">Sign out</button>  </div>
<button aria-label="Project actions" type="button" popovertarget="_JHUt-TD5TPe2QnQ9YIAPFQ">Options</button>  <div popover="auto" anchored dropdown id="_JHUt-TD5TPe2QnQ9YIAPFQ">
<button type="button" popovertarget="_JHUt-TD5TPe2QnQ9YIAPFQ" popovertargetaction="hide" autofocus>Rename</button><button type="button" disabled popovertarget="_JHUt-TD5TPe2QnQ9YIAPFQ" popovertargetaction="hide">Duplicate</button><hr><button type="button" danger popovertarget="_JHUt-TD5TPe2QnQ9YIAPFQ" popovertargetaction="hide">Delete</button>  </div>
</div>

```

## Invoker commands

`commandfor` names a target and `command` says what to do: `show-modal`, `close`, or `toggle-popover`. In a browser without invoker commands, the `commands` rule calls the matching method: `showModal()`, `close()` or `togglePopover()`. There is no constructor to learn.

In [ ]:
reg.rule(name='commands', sel='button[commandfor]', on={('&', 'click'): r"""
if ('commandForElement' in el) return;
const t = document.getElementById(el.getAttribute('commandfor')), c = el.getAttribute('command');
if (!t) return;
if (!['show-modal', 'close', 'toggle-popover'].includes(c)) return;
e.preventDefault();
if (c === 'show-modal') t.showModal();
else if (c === 'close') t.close();
else { t._anchor = el; t.togglePopover(); }"""},
    layer='components', doc='`command` and `commandfor` on engines without invoker commands.')

commands [components] button[commandfor]: `command` and `commandfor` on engines without invoker commands.

## Dialog

`Dialog` is the platform's `dialog` element, opened as a modal with `command="show-modal"`. We write no script for focus, Escape or the inert page: the browser moves focus into the dialog, closes it on Escape and makes the page behind it inert. A dialog has the popover's floating surface with the larger corner radius, `--radius1`, and its backdrop is the text colour, `--n2`, at 40% opacity. Both fade in and out over `--motion`. A dialog in the top layer still inherits the page's font and custom properties from its DOM parent. To close a dialog, use a button with `command="close"`, or a form with `method="dialog"`. A click on the backdrop closes a dialog only when the dialog has `closedby="any"`. Drawers use it, and confirmations don't.

In [ ]:
Dialog = hypelt('Dialog', layer='components', doc='A modal panel over an inert page; `drawer` slides it in from an edge.', css=r"""
--face: var(--n-2); background: var(--face); color: var(--n2);
border: 1px solid var(--n-1); border-radius: var(--radius1); padding: var(--space1); box-shadow: var(--floating);
inline-size: min(32rem, calc(100% - 2 * var(--space)));
translate: 0 var(--space-2);
&[open] { translate: none; @starting-style { translate: 0 var(--space-2); } }
&::backdrop { background: oklch(from var(--n2) l c h / .4); }
""")
Dialog.variant('drawer', r"""
--_slide: 100%; --drawer-size: 24rem; --face: var(--n-3); inset: 0; inset-inline-start: auto; overflow-y: auto;
block-size: 100dvh; max-block-size: none; inline-size: var(--drawer-size); max-inline-size: none; margin: 0;
border-radius: 0; border-block: 0; border-inline-end: 0; translate: var(--_slide) 0;
&:dir(rtl) { --_slide: -100%; }
&[drawer=start] { --_slide: -100%; inset-inline: 0 auto; border-inline: 0; border-inline-end: 1px solid var(--n-1); }
&[drawer=start]:dir(rtl) { --_slide: 100%; }
&[open] { translate: none; @starting-style { translate: var(--_slide) 0; } }
@media (width < 40rem) { &, &[drawer=start] { inline-size: 100%; border: 0; } }
""", layer='components', doc='Slides in from the end edge and runs the full height, `--drawer-size` wide, or the whole screen below 40rem, on the page colour, scrolling when its content is taller; `start` slides in from the other edge.')

Dialog.drawer [components] :is(dialog)[drawer]: Slides in from the end edge and runs the full height, `--drawer-size` wide, or the whole screen below 40rem, on the page colour, scrolling when its content is taller; `start` slides in from the other edge.

The `fill` rule opens a closed dialog when htmx swaps new content into it. A button that loads the dialog's body with `hx-get` therefore needs no `command`. The dialog opens only once the new content has arrived, and never shows stale content. When htmx swaps the dialog element itself, as when the server sends a closed dialog out of band to close it, the rule doesn't open it.

In [ ]:
Dialog.rule('fill', on={('&', ':after-swap'): "if (e.target !== el && !el.open) el.showModal()"}, layer='components',
    doc='A closed dialog opens when htmx swaps content into it; a swap of the dialog itself leaves it as it is.')

Dialog.fill [components] dialog: A closed dialog opens when htmx swaps content into it; a swap of the dialog itself leaves it as it is.

Here is a confirmation dialog with the button that opens it. `aria-labelledby` gives the dialog its heading's text as its accessible name. A heading inside a dialog doesn't name the dialog by itself. The content is an ordinary stack with a cluster of buttons.

In [ ]:
@example(Dialog)
def _dialog_example():
    id, title = unqid(), unqid()
    return Cluster(Button('Delete project', type='button', danger=True, command='show-modal', commandfor=id),
        Dialog(Stack(H2('Delete this project?', id=title), P('Everything in it is removed. This cannot be undone.'),
            Cluster(Button('Cancel', type='button', secondary=True, command='close', commandfor=id, autofocus=True),
                Button('Delete', type='button', danger=True, command='close', commandfor=id), gap='var(--space-2)')),
            id=id, aria_labelledby=title))

In [ ]:
Dialog.demo(height='16rem')

```html
<div cluster>
<button type="button" danger command="show-modal" commandfor="_ioNqwHCQT7isETNW_97goQ">Delete project</button><dialog aria-labelledby="_KnSB8gGWSDalquxfZNNd8A" id="_ioNqwHCQT7isETNW_97goQ">    <div stack>
      <h2 id="_KnSB8gGWSDalquxfZNNd8A">Delete this project?</h2>
      <p>Everything in it is removed. This cannot be undone.</p>
      <div cluster style="--gap: var(--space-2);">
<button type="button" secondary command="close" commandfor="_ioNqwHCQT7isETNW_97goQ" autofocus>Cancel</button><button type="button" danger command="close" commandfor="_ioNqwHCQT7isETNW_97goQ">Delete</button>      </div>
    </div>
</dialog></div>

```

## Drawer

A drawer is a dialog that slides in from an edge and is as tall as the viewport: `drawer=True` for the end edge, which is the right edge on a left-to-right page, and `drawer='start'` for the start edge. It is `--drawer-size` wide, 24rem unless the app sets the token. In a viewport narrower than 40rem it fills the screen, because a sliver of page beside it is no use on a phone. It scrolls when its content is taller than the screen. Its background is the page colour, `--n-3`, rather than the dialog's raised surface, because a drawer works more as a region of the page than as a floating panel, and cards inside it must stand out from it. Opening, closing, the backdrop and the inert page all work as for any dialog. With `closedby="any"`, a click outside the drawer closes it in browsers that support the attribute. In other browsers its button still closes it:

In [ ]:
demo(Cluster(Button('Filters', type='button', secondary=True, command='show-modal', commandfor='filters'),
    Button('Navigation', type='button', secondary=True, command='show-modal', commandfor='nav-drawer'),
    Dialog(Stack(H2('Filters', id='filters-title'), P('Narrow the list.'),
        Button('Done', type='button', command='close', commandfor='filters')),
        id='filters', aria_labelledby='filters-title', drawer=True, closedby='any'),
    Dialog(Stack(H2('Navigation', id='nav-title'), P('Sections of the app.'),
        Button('Close', type='button', secondary=True, command='close', commandfor='nav-drawer')),
        id='nav-drawer', aria_labelledby='nav-title', drawer='start', closedby='any')), height='12rem')

```html
<div cluster>
<button type="button" secondary command="show-modal" commandfor="filters">Filters</button><button type="button" secondary command="show-modal" commandfor="nav-drawer">Navigation</button><dialog aria-labelledby="filters-title" drawer closedby="any" id="filters">    <div stack>
      <h2 id="filters-title">Filters</h2>
      <p>Narrow the list.</p>
<button type="button" command="close" commandfor="filters">Done</button>    </div>
</dialog><dialog aria-labelledby="nav-title" drawer="start" closedby="any" id="nav-drawer">    <div stack>
      <h2 id="nav-title">Navigation</h2>
      <p>Sections of the app.</p>
<button type="button" secondary command="close" commandfor="nav-drawer">Close</button>    </div>
</dialog></div>

```

## Tooltip

`Tooltip` pairs a control with a short, supplementary description. It adds the description's id to the control's `aria-describedby`, keeping any id already there, and returns both in one inline `span`. The control still needs its own name; a tooltip is not a substitute for a label.

The description is real markup, shown by CSS alone. It shows while the pointer is over the control or the description, or while the control has keyboard focus. Once neither holds, it waits 150ms and then fades out over `--motion`. The wait lets the pointer move from the control onto the description. A click that focuses the control does not show it. It appears at once. Moving from one control to another hides the first description and shows the second at once, with no fade, as the platform's own tooltips do. The control is the description's anchor: anchor positioning centres the description above the control, or below it when there is no room above. In a browser without anchor positioning, the description sits above the control, aligned to its left edge. It is not a popover, so it is not in the top layer: an ancestor with a `transform` or a `filter` becomes its containing block and can clip it. Put interactive content in a `Popover`, not a tooltip.

In [ ]:
_tooltip_css = r"""
display: inline-block; position: relative;
> [role=tooltip] {
  --face: var(--n2); background: var(--face); color: var(--n-3); border: 0;
  padding: var(--space-3) var(--space-2); border-radius: var(--radius-1);
  inline-size: max-content; max-inline-size: min(20rem, calc(100vw - 2 * var(--space))); overflow-wrap: anywhere;
  font-size: var(--text-1); font-weight: var(--weight); line-height: var(--leading);
  position: fixed; inset: auto; position-area: block-start; position-try-fallbacks: flip-block; margin-block-end: 4px; z-index: 1;
  visibility: hidden; opacity: 0;
  transition: opacity var(--motion) var(--ease) 150ms, visibility 0s linear calc(var(--motion) + 150ms);
}
&:is(:hover, :has(:focus-visible)) > [role=tooltip] { visibility: visible; opacity: 1; transition: none; }
:root:has([tooltip]:is(:hover, :has(:focus-visible))) &:not(:hover, :has(:focus-visible)) > [role=tooltip] { transition: none; }
@supports not (position-area: center) {
  > [role=tooltip] { position: absolute; inset: auto auto 100% 0; }
}
"""

@hypelt(sel='[tooltip]', layer='components', css=_tooltip_css)
def Tooltip(
    trigger, # A control with its own accessible name
    text,    # Supplementary, non-interactive content
    id=None, # Description id; generated when omitted
    **attrs, # Attributes for the description
):
    "A control and its supplementary description, shown on hover or keyboard focus"
    id = id or unqid()
    describedby = ' '.join(filter(None, [trigger.get('aria-describedby'), id]))
    trigger(aria_describedby=describedby, style=' '.join(filter(None, [trigger.get('style'), f'anchor-name: --{id};'])))
    tip = Div(text, role='tooltip', id=id, **attrs)(style=' '.join(filter(None, [attrs.get('style'), f'position-anchor: --{id};'])))
    return Span(trigger, tip, tooltip=True)


In [ ]:
@example(Tooltip)
def _tooltip_example():
    return Cluster(Tooltip(Button('Archive', type='button', secondary=True), 'Move to the archive'),
        Tooltip(Button('Share', type='button', secondary=True), 'Copy a link'),
        Tooltip(Button('Delete', type='button', danger=True), 'This cannot be undone'))

In [ ]:
Tooltip.demo(height='7rem')

```html
<div cluster>
<span tooltip><button type="button" secondary aria-describedby="_Bh5hfdE_RW20VRL1gAcLmg" style="anchor-name: --_Bh5hfdE_RW20VRL1gAcLmg;">Archive</button>    <div role="tooltip" id="_Bh5hfdE_RW20VRL1gAcLmg" style="position-anchor: --_Bh5hfdE_RW20VRL1gAcLmg;">Move to the archive</div>
</span><span tooltip><button type="button" secondary aria-describedby="_FhXZSSNmQoWkmr-kdQBKqw" style="anchor-name: --_FhXZSSNmQoWkmr-kdQBKqw;">Share</button>    <div role="tooltip" id="_FhXZSSNmQoWkmr-kdQBKqw" style="position-anchor: --_FhXZSSNmQoWkmr-kdQBKqw;">Copy a link</div>
</span><span tooltip><button type="button" danger aria-describedby="_JRsm75xLTbqWU1hJaQFlYg" style="anchor-name: --_JRsm75xLTbqWU1hJaQFlYg;">Delete</button>    <div role="tooltip" id="_JRsm75xLTbqWU1hJaQFlYg" style="position-anchor: --_JRsm75xLTbqWU1hJaQFlYg;">This cannot be undone</div>
</span></div>

```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()